# Deliver Death (Mercenary)

Deliver Death is a melee finisher on a 45 s cooldown. It hits an adjacent enemy for a multiple
of the Mercenary's **weapon value**. The multiplier is much larger when the target is below 25%
HP. There is no hit roll and no dodge, so only armor and the target's HP threshold matter.

**Rebalanced.** The skill scale used to be `clamp(50 + p/2, 50, 150)`. It reached 150% at
modified skill 200 and was flat after that, so the last third of a top-rank Mercenary's skill
growth (mod 200–300) added nothing. The scale now uses the shared mod-300 knee,
`skill_knee_300(p, 50, 150)`: 150% at modified skill 300, then half the slope with no hard limit.
Endgame damage at 300 is unchanged. Damage below 300 is lower: at the Mercenary raise cap of 120
the scale drops from 110% to 90%. The same scale drives the Blade Dance active. The "before"
curves below show the old scale.

| | |
|---|---|
| Class | Mercenary |
| Target | Selected enemy; must be adjacent (including diagonals) |
| Cost | 75 endurance |
| Cooldown | 45 s (`add_skill_cooldown`), started even when the blow glances off |
| Landing roll | none: no melee hit table, no dodge |
| Skill scale | `skill_knee_300(p, 50, 150)` %: 90% at 120, 150% at 300, 158% at 350; no hard limit |
| Damage | `do_hurt(dam, 0)`: melee damage, reduced by armor, at ¼ HP per point |
| Source | `server/src/driver/skill/mercenary.rs::skill_deliver_death`, `weapon_skill_scale_pct`; `server/src/driver/skill/common.rs::skill_knee_300` |

---

# Formula

With the caster's weapon value $W = \max(1, \text{weapon})$, the modified Deliver Death skill
$p$ (negative counts as 0) and the target's HP fraction $h = \lfloor a\_hp / \text{max\_hp} \rfloor$
in permille:

$$
\text{base} = \begin{cases}
15W + \text{rand}(6W) & h < 250 \quad\text{(execute)}\\
6W + \text{rand}(3W) & \text{otherwise}
\end{cases}
\qquad
\text{scale} = \begin{cases}
50 + \left\lfloor \dfrac{100\,p}{300} \right\rfloor & p \le 300\\[2ex]
150 + \left\lfloor \dfrac{100\,(p - 300)}{600} \right\rfloor & p > 300
\end{cases}\%
$$

$$
\text{dam} = \left\lfloor \text{base}\cdot\text{scale}/100 \right\rfloor
\qquad
\text{HP removed} = \max(0,\ \text{dam} - \text{armor}) / 4
$$

The skill does **not** go through immunity or the race modifier. It only sets the scale, which is
linear from 50% at skill 0 to 150% at skill 300 (+1% per 3 points), then +1% per 6 points.

Before: $\text{scale} = \operatorname{clamp}(50 + \lfloor p/2 \rfloor,\ 50,\ 150)$, which reached
150% at $p = 200$ and stayed there.

| Quantity | Now | Before |
|---|---|---|
| Scale at $p = 0$ | 50% | 50% |
| Scale at the raise caps (105 / 120) | 85% / 90% | 102% / 110% |
| 100% reached at | $p = 150$ | $p = 100$ |
| Scale at $p = 200$ | 116% | 150% (ceiling) |
| Scale at the endgame band (275 / 300) | 141% / 150% | 150% / 150% |
| Scale at $p = 350$ / 450 | 158% / 175% | 150% / 150% |
| Hard limit | none | 150% from $p = 200$ |
| Execute threshold | target below 25.0% HP (permille, truncated) | same |

---

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import mag_model as mm

mm.apply_style()
P = mm.SKILL_RANGE
COOLDOWN_S = 45
END_COST = 75

# Model

In [ ]:
def scale_pct(p):
    """`weapon_skill_scale_pct`: `skill_knee_300(p, 50, 150)`."""
    return mm.knee_300(p, 50, 150)


def old_scale_pct(p):
    """Before the rebalance: `clamp(50 + p/2, 50, 150)`."""
    return mm.clamp(50 + mm.idiv(p, 2), 50, 150)


def dd_rolls(weapon, p, execute, scale=scale_pct):
    """All equally likely `dam` values (before armor) for one cast."""
    w = max(1, weapon)
    lo, n = (15 * w, 6 * w) if execute else (6 * w, 3 * w)
    base = lo + np.arange(n)
    return mm.idiv(base * scale(p), 100)


def dd_expected_hp(weapon, p, execute, armor=0, scale=scale_pct):
    p = np.atleast_1d(p)
    out = np.array([np.mean(mm.hurt_hp(dd_rolls(weapon, int(x), execute, scale), armor)) for x in p])
    return mm._scalar(out if out.size > 1 else out[0])


def dd_min_hp(weapon, p, execute, armor=0, scale=scale_pct):
    p = np.atleast_1d(p)
    out = np.array([mm.hurt_hp(dd_rolls(weapon, int(x), execute, scale).min(), armor) for x in p])
    return mm._scalar(out if out.size > 1 else out[0])


def swing_expected_hp(weapon, strength, armor=0):
    """A normal landed melee hit at Δ = 0 (no table bonus), for comparison."""
    return np.mean(mm.hurt_hp(mm.melee_raw_rolls(weapon, strength), armor))

# Validation

- `weapon_skill_scale_reaches_one_and_a_half_at_skill_300`: 0 → 50, 150 → 100, 300 → 150,
  450 → 175.
- `deliver_death_requires_endurance`: `a_end` 74,999 is refused (75 endurance = 75,000).
- `deliver_death_is_blocked_while_recharging`: no endurance is spent during the cooldown.

In [ ]:
assert [int(scale_pct(p)) for p in (0, 150, 300, 450)] == [50, 100, 150, 175]
assert 74_999 < END_COST * 1000
assert [int(scale_pct(p)) for p in (105, 120, 200, 275, 350)] == [85, 90, 116, 141, 158]
assert [int(old_scale_pct(p)) for p in (0, 100, 105, 120, 200, 400)] == [50, 100, 102, 110, 150, 150]

# Execute at W 40, skill 120: base 600..839, x0.90 (before x1.10).
r = dd_rolls(40, 120, True)
assert r.min() == 540 and r.max() == 755 and len(r) == 240
assert abs(dd_expected_hp(40, 120, True) - 161.775) < 1e-9
r = dd_rolls(40, 120, True, old_scale_pct)
assert r.min() == 660 and r.max() == 922
assert dd_expected_hp(40, 120, True, scale=old_scale_pct) == 197.75
# Same damage at 300, more past it.
assert dd_expected_hp(40, 300, True) == dd_expected_hp(40, 300, True, scale=old_scale_pct) == 269.75
print("Deliver Death model OK")

# Plots

In [ ]:
fig, ax = plt.subplots()
ax.step(P, scale_pct(P), where="post", lw=2.2, label="weapon_skill_scale_pct")
ax.step(P, old_scale_pct(P), where="post", color="grey", lw=1.2, ls=":", label="before")
mm.skill_axis(ax, "damage scale (%)", "Deliver Death / Blade Dance skill scale")
mm.mark_caps(ax, extra=[(200, "old 150% at 200"), (300, "150% at 300")])
ax.legend(loc="lower right")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, execute, title in ((axes[0], True, "Execute (target < 25% HP)"),
                           (axes[1], False, "Normal (target ≥ 25% HP)")):
    for wv in mm.WV_SWEEP:
        line, = ax.plot(P, dd_expected_hp(wv, P, execute), label=f"weapon value {wv}")
        if wv == 40:
            ax.plot(P, dd_expected_hp(wv, P, execute, scale=old_scale_pct), color=line.get_color(),
                    lw=1.2, ls=":", label="before, weapon value 40")
    mm.skill_axis(ax, "expected HP removed (armor 0)", title)
    mm.mark_caps(ax)
    ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.88), fontsize=8)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for armor in mm.ARMOR_SWEEP:
    line, = ax.plot(P, dd_expected_hp(40, P, True, armor), label=f"armor {armor}")
    if armor == 0:
        ax.plot(P, dd_expected_hp(40, P, True, armor, scale=old_scale_pct), color=line.get_color(),
                lw=1.2, ls=":", label="before, armor 0")
mm.skill_axis(ax, "expected HP removed", "Execute vs armor (weapon value 40)")
mm.mark_caps(ax)
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.88), fontsize=8)

# Execute damage measured in normal landed hits of the same weapon.
ax = axes[1]
for wv in (20, 40, 60):
    swing = swing_expected_hp(wv, 80)
    line, = ax.plot(P, dd_expected_hp(wv, P, True) / swing, label=f"execute, WV {wv}")
    ax.plot(P, dd_expected_hp(wv, P, False) / swing, color=line.get_color(), ls="--", label=f"normal, WV {wv}")
    if wv == 40:
        ax.plot(P, dd_expected_hp(wv, P, True, scale=old_scale_pct) / swing, color=line.get_color(),
                lw=1.2, ls=":", label="before, execute WV 40")
mm.skill_axis(ax, "equivalent landed hits (STR 80, armor 0)", "One cast in normal melee hits")
mm.mark_caps(ax)
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.88), fontsize=8)
plt.tight_layout()

Measured in swings, Deliver Death favours high weapon value. The finisher scales only with WV,
while a swing also carries the flat `rand(6) + rand(STR/2)` term. With STR 80, an execute is worth
7.5 hits with WV 20 and 11.7 hits with WV 60 at skill 120 (before: 9.2 and 14.3). At skill 300 it
is worth 12.5 and 19.5 hits, the same as before.

In [ ]:
# Largest max HP an execute is guaranteed / expected to kill from just under 25% HP.
fig, ax = plt.subplots()
for wv in (20, 40, 60):
    line, = ax.plot(P, 4 * dd_expected_hp(wv, P, True), label=f"WV {wv}, average roll")
    ax.plot(P, 4 * dd_min_hp(wv, P, True), color=line.get_color(), ls="--", label=f"WV {wv}, minimum roll")
    if wv == 40:
        ax.plot(P, 4 * dd_expected_hp(wv, P, True, scale=old_scale_pct), color=line.get_color(),
                lw=1.2, ls=":", label="before, WV 40 average")
mm.skill_axis(ax, "target max HP (armor 0)", "Largest target an execute finishes from 24.9% HP")
mm.mark_caps(ax)
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.88), fontsize=8)

# Observations

- **Endgame damage is unchanged at 300 and now keeps growing.** With weapon value 40 at skill
  300, an execute removes 270 HP on average (armor 0), the same as before. At the start of the
  endgame band (275) it removes 254 HP (before: 270, already flat). At 350 it removes 284 HP,
  +5.3% over 300; before, skill past 200 added nothing.
- **Mid-game damage is lower.** At the Mercenary raise cap of 120 the scale is 90% (before:
  110%). The WV-40 execute drops from 198 to 162 HP (−18%), and the minimum roll from 165 to
  135 HP. At 105 it drops from 183 to 153 HP (−17%). The new scale is below the old one at every
  skill from 4 to 299, and reaches 100% only at 150.
- **Still a strong execute at almost no cost.** At skill 120 the average execute finishes any
  target with up to about 650 max HP from just under 25% (540 guaranteed; before: 790 and 660).
  At skill 300 that is 1,080 (900 guaranteed); at 350 it is about 1,140 (950 guaranteed). With no
  hit roll and no dodge, it is the most reliable melee damage the Mercenary has.
- **The non-execute branch is still worth casting.** Above 25% HP the same character deals 67 HP
  at skill 120 (before: 82), about 4.3 hits' worth, and 112 HP at skill 300. The cooldown is the
  same, so the best play is to hold it for the execute. Casting it early still beats letting it
  sit unused for a whole fight.
- **Over a 45 s cooldown it adds roughly +8% to +31% melee DPS.** At 2.025 swings/s and 60% to
  hit, a Mercenary lands about 55 hits per 45 s. At skill 120 one execute adds 10.3 more (+19%),
  one normal cast 4.3 (+8%). At skill 300 that is 17.1 (+31%) and 7.1 (+13%). Across WV 20–60
  and skill 105–300, an execute is worth 7.1 to 19.5 hits.
- **Skill now matters as much as weapon value.** Going from 105 to 120 adds 6% damage (85% →
  90%). Going from 120 to 300 adds 67% (90% → 150%). Going from WV 40 to 60 adds about 50%.
  Before, the scale was 102% to 110% at the raise caps and flat from 200, so skill past 200 was
  worth nothing. Immunity and race are still ignored.
- **Armor is a small, flat tax.** The finisher hits in one big chunk, so armor is subtracted once.
  Armor 40 lowers the WV-40 execute from 162 to 152 HP at skill 120 (−6%) and from 270 to 260 HP
  at 300 (−4%). The normal branch loses 15% at 120 and 9% at 300. The lower mid-game scale makes
  armor slightly more relevant than before.
- **Endurance is spent and the cooldown starts even when the blow glances off.** On a
  high-armor target the cast can do 0 damage and still cost 75 endurance and 45 s.
- **Open question: a raise-capped Mercenary is below 100% scale.** At 120 the finisher deals 90%
  of its nominal weapon multiple, and needs +30 modified skill from gear or buffs to reach 100%.
  If the `15W` / `6W` multiples were tuned for mid-game, they could be raised slightly, or the
  curve could start above 50%.